In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path("data")
df = pd.read_parquet(DATA / "model_table.parquet")

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["TARGET"], random_state=42
)

print("Train:", train_df.shape, " default %:", round(train_df["TARGET"].mean() * 100, 2))
print("Test: ", test_df.shape,  " default %:", round(test_df["TARGET"].mean() * 100, 2))

train_df[["SK_ID_CURR"]].to_parquet(DATA / "train_ids.parquet", index=False)
test_df[["SK_ID_CURR"]].to_parquet(DATA / "test_ids.parquet", index=False)

Train: (246008, 139)  default %: 8.07
Test:  (61503, 139)  default %: 8.07


In [3]:
import numpy as np

income_cap = train_df["AMT_INCOME_TOTAL"].quantile(0.999)   # from TRAIN only

def clean(d, income_cap):
    d = d.copy()
    d["is_pensioner_or_unemployed"] = (d["DAYS_EMPLOYED"] == 365243).astype(int)
    d.loc[d["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    d["ORGANIZATION_TYPE"]  = d["ORGANIZATION_TYPE"].replace("XNA", np.nan)
    d["CODE_GENDER"]        = d["CODE_GENDER"].replace("XNA", np.nan)
    d["NAME_FAMILY_STATUS"] = d["NAME_FAMILY_STATUS"].replace("Unknown", np.nan)
    d["AMT_INCOME_TOTAL"]   = d["AMT_INCOME_TOTAL"].clip(upper=income_cap)
    d["age_years"]          = -d["DAYS_BIRTH"] / 365.25
    d["credit_to_income"]   = d["AMT_CREDIT"]  / d["AMT_INCOME_TOTAL"]
    d["annuity_to_income"]  = d["AMT_ANNUITY"] / d["AMT_INCOME_TOTAL"]
    return d

train_df = clean(train_df, income_cap)
test_df  = clean(test_df,  income_cap)

print("Income cap (99.9th pct of train):", round(income_cap))
print("Max DAYS_EMPLOYED now:", train_df["DAYS_EMPLOYED"].max())
print("Pensioner flag in train:", train_df["is_pensioner_or_unemployed"].sum())
print("Shape train/test:", train_df.shape, test_df.shape)

Income cap (99.9th pct of train): 900000
Max DAYS_EMPLOYED now: 0.0
Pensioner flag in train: 44256
Shape train/test: (246008, 143) (61503, 143)


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

drop = ["SK_ID_CURR", "TARGET"]
num_cols = [c for c in train_df.select_dtypes(include="number").columns if c not in drop]

X_train, y_train = train_df[num_cols], train_df["TARGET"]
X_test,  y_test  = test_df[num_cols],  test_df["TARGET"]

baseline = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
baseline.fit(X_train, y_train)

p_test = baseline.predict_proba(X_test)[:, 1]

print("Features used:", len(num_cols))
print("ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))
print("PR-AUC: ", round(average_precision_score(y_test, p_test), 4))
print("PR-AUC of random guessing:", round(y_test.mean(), 4))

Features used: 125
ROC-AUC: 0.7525
PR-AUC:  0.2304
PR-AUC of random guessing: 0.0807


In [5]:
train_df.to_parquet(DATA / "train_clean.parquet", index=False)
test_df.to_parquet(DATA / "test_clean.parquet", index=False)

pd.DataFrame({"SK_ID_CURR": test_df["SK_ID_CURR"],
              "TARGET": y_test,
              "p_baseline": p_test}).to_parquet(DATA / "test_preds_baseline.parquet", index=False)
print("Saved.")

Saved.


## Notebook 05: baseline model

I split the data 80/20 before any cleaning, keeping 8.07% defaults in both parts.

Cleaning, using what I found earlier:
- 365,243 in DAYS_EMPLOYED became blank, plus a pensioner flag.
- XNA and Unknown became blank.
- Income capped at 900,000 (99.9th percentile, from training data only).
- Added age in years, loan-to-income, and payment-to-income.

Baseline: Logistic Regression on 125 numeric features.
- ROC-AUC: 0.7525
- PR-AUC: 0.2304 (random guessing: 0.0807)

This is the score the next model has to beat. Gender was not used as a feature.